# Timeseries REMA Analysis Notebook - Comparison of Same year vs Next year DEM on a winter Scene
- Purpose of this notebook is to compare a winter scene (June 2021) with the REMA 2021 dem and REMA 2022 DEM
- As the DEMs are produced for the end of each year (summer), this scene sits between two DEMs temporally 
- Significant difference could indicate the annual DEM is too sparse

In [ ]:
import fsspec
import xarray as xr
import rioxarray
from dotenv import load_dotenv
import re
import pandas as pd
import numpy as np
from scipy.ndimage import uniform_filter
import geopandas as gpd
from dea_tools.plotting import xr_animation
from dea_tools.temporal import xr_optical_flow, xr_regression # !pip install dea-tools==0.4.8dev13
from PIL import Image, ImageSequence
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import os
import json
from shapely.geometry import Polygon, mapping
import contextily as ctx
import cv2

In [ ]:
# extra reguirements de-sar-sample-data
#!pip install dea-tools==0.4.8dev13
#!pip install contextily
#!pip install opencv-python-headless

## Set envrionment credentials for AWS access

In [ ]:
load_dotenv('/home/ec2-user/sar-pipeline/.env')

## Select the burst to assess

In [ ]:
# t007_014550_iw2 (scene 1) - slope and aspect
# t007_014551_iw2 (scene 1) - slope and aspect
# t007_014549_iw1 (scene 1) - elevation
# t007_014543_iw2 (scene 2) - elevation
burst = "t007_014551_iw2"
burst_shape = f"burst_shapefiles/{burst}.json"
burst_shape = gpd.read_file(burst_shape)

## Create a smaller ROI within burst for zoom

In [ ]:
burst_roi = True # if true, assess a zoomed area centered around below. Else assess whole burst.
if burst_roi:
    side_length = 10_000
    # center of the burst
    burst_roi_centers = {
        't007_014550_iw2' : (-1514900,-640900),
        't007_014551_iw2' : (-1515100-42_000,-660600+5_000),
        't007_014549_iw1' : (-1482400,-623700),
        't007_014543_iw2' : (-1514772+20_000, -508648),
    }

    # all areas of interest
    burst_roi_aoi_centres = {
        't007_014550_iw2' : [(-1514900,-640900)],
        't007_014551_iw2' : [(-1515100,-660600), (-1515100-42_000,-660600+5_000)],
        't007_014549_iw1' : [(-1482400,-623700)],
        't007_014543_iw2' : [(-1514772,-508648),(-1514772+20_000, -508648)],
    }


def save_square_geojson(center_x, center_y, side_length, filename):
    """
    Create a square polygon around a centre point (EPSG:3031) and save as GeoJSON.

    Parameters
    ----------
    center_x : float
        X coordinate (EPSG:3031)
    center_y : float
        Y coordinate (EPSG:3031)
    side_length : float
        Length of square's side (same units as EPSG:3031, e.g. metres)
    filename : str
        Output filename for the GeoJSON file

    Returns
    -------
    dict
        A GeoJSON FeatureCollection dictionary
    """
    
    half = side_length / 2.0

    # Define square corners (clockwise or counter-clockwise is fine)
    square = Polygon([
        (center_x - half, center_y - half),
        (center_x + half, center_y - half),
        (center_x + half, center_y + half),
        (center_x - half, center_y + half),
        (center_x - half, center_y - half),
    ])

    feature = {
        "type": "Feature",
        "geometry": mapping(square),
        "properties": {
            "center_x": center_x,
            "center_y": center_y,
            "side_length": side_length,
            "crs": "EPSG:3031"
        }
    }

    feature_collection = {
        "type": "FeatureCollection",
        "features": [feature]
    }

    # Save to file
    with open(filename, "w") as f:
        json.dump(feature_collection, f, indent=2)

    return feature_collection

if burst_roi:
    roi_x, roi_y = burst_roi_centers[burst]
    burst_roi_shape = f'burst_shapefiles/{burst}_{roi_x}_{roi_y}.json'
    save_square_geojson(roi_x, roi_y, side_length, burst_roi_shape)
    burst_roi_shape = gpd.read_file(burst_roi_shape)
    burst_roi_shape = burst_roi_shape.set_crs(epsg=3031, inplace=False,allow_override=True)
    
    # plot to see overlap
    fig, ax = plt.subplots(figsize=(8,8))
    burst_shape_3031 = burst_shape.to_crs(epsg=3031)
    burst_roi_shape.plot(ax=ax, color='red', alpha=0.5, edgecolor='black', label="ROI")
    burst_shape_3031.plot(ax=ax, color='blue', alpha=0.5, edgecolor='black', label=f"{burst}")
    ax.legend()
    ax.set_title(f"Burst : {burst} and ROI overlap")
    plt.show()
    

## Functions

In [ ]:
# Adapted from https://stackoverflow.com/questions/39785970/speckle-lee-filter-in-python
def lee_filter(img, size):
    """
    Applies the Lee filter to reduce speckle noise in an image.

    Parameters:
    img (ndarray): Input image to be filtered.
    size (int): Size of the uniform filter window.

    Returns:
    ndarray: The filtered image.
    """
    img_mean = uniform_filter(img, size)
    img_sqr_mean = uniform_filter(img**2, size)
    img_variance = img_sqr_mean - img_mean**2

    overall_variance = np.var(img)

    img_weights = img_variance / (img_variance + overall_variance)
    img_output = img_mean + img_weights * (img - img_mean)
    return img_output


# Define a function to apply the Lee filter to a DataArray
def apply_lee_filter(data_array, size=7):
    """
    Applies the Lee filter to the provided DataArray.

    Parameters:
    data_array (xarray.DataArray): The data array to be filtered.
    size (int): Size of the uniform filter window. Default is 7.

    Returns:
    xarray.DataArray: The filtered data array.
    """
    data_array_filled = data_array.fillna(0)
    filtered_data = xr.apply_ufunc(
        lee_filter,
        data_array_filled,
        kwargs={"size": size},
        input_core_dims=[["y", "x"]],
        output_core_dims=[["y", "x"]],
        dask_gufunc_kwargs={"allow_rechunk": True},
        vectorize=True,
        dask="parallelized",
        output_dtypes=[data_array.dtype],
    )
    filtered_data_masked = xr.where(np.isnan(data_array), np.nan, filtered_data)

    return filtered_data_masked


def slope_aspect(dem, dx=1.0, dy=1.0):
    """
    Compute slope and aspect from a 2D DEM array, handling nodata values.
    
    Parameters
    ----------
    dem : np.ndarray
        2D elevation array (NaNs represent nodata).
    dx : float
        Spatial resolution in x-direction.
    dy : float
        Spatial resolution in y-direction.
    
    Returns
    -------
    slope : np.ndarray
        Slope in degrees.
    aspect : np.ndarray
        Aspect in degrees (0 = North, clockwise).
    """
    # Mask invalid values
    dem = np.where(np.isfinite(dem), dem, np.nan)
    
    # Compute gradients
    dz_dy, dz_dx = np.gradient(dem, dy, dx)
    
    # Replace NaN gradients with 0 to avoid NaNs in slope/aspect
    dz_dx = np.nan_to_num(dz_dx)
    dz_dy = np.nan_to_num(dz_dy)
    
    # Slope in degrees
    slope = np.arctan(np.sqrt(dz_dx**2 + dz_dy**2)) * 180 / np.pi
    
    # Aspect in degrees
    aspect = np.arctan2(-dz_dx, -dz_dy) * 180 / np.pi
    aspect = np.where(np.isnan(dem), np.nan, (aspect + 360) % 360)
    
    return slope, aspect


def slope_aspect_timeseries(ds_dem, dx=1.0, dy=1.0):
    """
    Apply slope_aspect function to an xarray DataArray over time,
    preserving NaNs for nodata values.
    
    Parameters
    ----------
    ds_dem : xarray.DataArray
        DEM time series with dimensions ('time', 'y', 'x')
    dx, dy : float
        Spatial resolution
    
    Returns
    -------
    slopes : xarray.DataArray
        Slope for each timestep
    aspects : xarray.DataArray
        Aspect for each timestep
    """
    slopes, aspects = xr.apply_ufunc(
        slope_aspect,
        ds_dem,
        dx,
        dy,
        input_core_dims=[["y", "x"], [], []],
        output_core_dims=[["y", "x"], ["y", "x"]],
        vectorize=True,
        dask="parallelized",
        output_dtypes=[ds_dem.dtype, ds_dem.dtype],
    )
    
    slopes = xr.DataArray(slopes, coords=ds_dem.coords, dims=ds_dem.dims, name="slope")
    aspects = xr.DataArray(aspects, coords=ds_dem.coords, dims=ds_dem.dims, name="aspect")
    
    return slopes, aspects

def keep_s3_prefix(file_list):
    return [f if f.startswith("s3://") else f"s3://{f}" for f in file_list]

# Extract timestamps from filenames (e.g., 20170405T050842)
def extract_time(fp):
    match = re.search(r"(\d{8}T\d{6})", fp)
    return pd.to_datetime(match.group(1)) if match else None

def load_xr_dataset(file_list):
    datasets = []
    for fp in file_list:
        da = rioxarray.open_rasterio(
            fp,
            masked=True,
            chunks=True,
        )
        da = da.squeeze().expand_dims(time=[extract_time(fp)])  # add time dim
        datasets.append(da)

        # Combine along the time dimension
    return xr.concat(datasets, dim="time")  # dims: ('time', 'y', 'x')

def preprocess_nrb_xr_dataset(ds_nrb, burst_shape, cal = 'gamma0'):
    # Open each file lazily with rioxarray and assign a time coordinate
    ds_nrb.name = f"HH_{cal}"
    # trim the dataset withg burst shapefule
    burst_shape = burst_shape.to_crs(ds_nrb.rio.crs)
    ds_nrb = ds_nrb.rio.clip(burst_shape.geometry, crs=ds_nrb.rio.crs, drop=True)
    ds_nrb.odc.assign_crs(crs='EPSG:3031')
    # Apply Lee filter directly on the DataArray
    ds_nrb[f"HH_{cal}_filtered"] = apply_lee_filter(ds_nrb, size=5)
    # Convert to dB only for positive values
    ds_nrb[f"HH_{cal}_db"] = xr.where(
        ds_nrb.to_dataset(name=f'HH_{cal}')[f'HH_{cal}'] > 0,
        10 * np.log10(ds_nrb.to_dataset(name=f'HH_{cal}')[f'HH_{cal}']),
        np.nan
    )
    ds_nrb[f"HH_{cal}_filtered_db"] = 10 * np.log10(ds_nrb[f'HH_{cal}_filtered'])
    return ds_nrb

def preprocess_dem_xr_dataset(ds_dem, burst_shape):
    ds_dem.name = "dem"
    burst_shape = burst_shape.to_crs(ds_dem.rio.crs)
    ds_dem = ds_dem.rio.clip(burst_shape.geometry, crs=ds_dem.rio.crs, drop=True)
    ds_dem.odc.assign_crs(crs='EPSG:3031')
    ds_dem = ds_dem.to_dataset(name="elevation")
    slopes, aspects = slope_aspect_timeseries(ds_dem['elevation'], dx=20, dy=20)
    ds_dem = ds_dem.assign({
        "slope": slopes,
        "aspect": aspects
    })
    return ds_dem


## Load in the xarrays

In [ ]:
# define the convention, gamma0 and beta0 in different folders
cal = 'beta0'
cal = 'gamma0'

In [ ]:
if not burst_roi:
    results_folder = f'timeseries-results/{cal}/{burst}'
else:
    results_folder = f'timeseries-results/{cal}/{burst}_{roi_x}_{roi_y}'
os.makedirs(results_folder, exist_ok=True)

In [ ]:
# Create S3 filesystem (anonymous access)
fs = fsspec.filesystem("s3", anon=True)

# Base S3 folder (public)
if cal == 'gamma0':
    rema_same_year_path = f"s3://deant-data-public-dev/experimental/rema-timeseries-v0.3-assessment/REMA_10_TIMESERIES_WINTER_SAME_YEAR_DEM/ga_s1_nrb_iw_hh_0/{burst}/"
    rema_next_year = f"s3://deant-data-public-dev/experimental/rema-timeseries-v0.3-assessment/REMA_10_TIMESERIES_WINTER_NEXT_YEAR_DEM/ga_s1_nrb_iw_hh_0/{burst}/"
if cal == 'beta0':
    rema_same_year_path = ...
    rema_next_year = ...

In [ ]:
# change static = same year, timeseries = next year

In [ ]:
# Find all GeoTIFFs recursively
same_year_nrb_tif_files = fs.glob(f"{rema_same_year_path}**/*HH-{cal}.tif")
same_year_dem_tif_files = fs.glob(f"{rema_same_year_path}**/*digital-elevation-model.tif")
next_year_nrb_tif_files = fs.glob(f"{rema_next_year}**/*HH-{cal}.tif")
next_year_dem_tif_files = fs.glob(f"{rema_next_year}**/*digital-elevation-model.tif")

# Ensure each file keeps the s3:// prefix (sometimes fsspec strips it)
same_year_nrb_tif_files = keep_s3_prefix(same_year_nrb_tif_files)
same_year_dem_tif_files = keep_s3_prefix(same_year_dem_tif_files)
next_year_nrb_tif_files = keep_s3_prefix(next_year_nrb_tif_files)
next_year_dem_tif_files = keep_s3_prefix(next_year_dem_tif_files)

print(f"Found {len(same_year_nrb_tif_files)} same_year NRB TIFs:")
print(f"Found {len(same_year_dem_tif_files)} same_year DEM TIFs:")
print(f"Found {len(next_year_nrb_tif_files)} next_year NRB TIFs:")
print(f"Found {len(next_year_dem_tif_files)} next_year DEM TIFs:")

# get the shape for trimming the xarray
clip_shape = burst_roi_shape if burst_roi else burst_shape

# load and preprocess the nrb datasets
ds_same_year_nrb = load_xr_dataset(same_year_nrb_tif_files)
ds_same_year_nrb = preprocess_nrb_xr_dataset(ds_same_year_nrb, clip_shape, cal=cal)
ds_next_year_nrb = load_xr_dataset(next_year_nrb_tif_files)
ds_next_year_nrb = preprocess_nrb_xr_dataset(ds_next_year_nrb, clip_shape, cal=cal)

# load in the dem datasets
ds_same_year_dem = load_xr_dataset(same_year_dem_tif_files)
ds_same_year_dem = preprocess_dem_xr_dataset(ds_same_year_dem, clip_shape)
ds_next_year_dem = load_xr_dataset(next_year_dem_tif_files)
ds_next_year_dem = preprocess_dem_xr_dataset(ds_next_year_dem, clip_shape)

## Compute differences

In [ ]:
# Compute difference at each timestep
nrb_diff_ts = ds_next_year_nrb[f'HH_{cal}_filtered_db'] - ds_same_year_nrb[f'HH_{cal}_filtered_db']
nrb_diff_ds = xr.Dataset({f"HH_{cal}_filtered_db_diff": nrb_diff_ts})

# Compute difference at each timestep
dem_diff_ts = ds_next_year_dem.elevation - ds_same_year_dem.elevation
dem_diff_ds = xr.Dataset({"elevation_m_diff": dem_diff_ts})

In [ ]:
if burst_roi:
    # load and preprocess the full burst nrb datasets
    ds_same_year_nrb_full_burst = load_xr_dataset(same_year_nrb_tif_files)
    ds_same_year_nrb_full_burst = preprocess_nrb_xr_dataset(ds_same_year_nrb_full_burst, burst_shape, cal=cal)
    da_same_full_burst = ds_same_year_nrb_full_burst.isel(time=0)[f'HH_{cal}_filtered_db']

    # load and preprocess the full burst DEM datasets - get the diff for plotting
    ds_same_year_dem_full_burst = load_xr_dataset(same_year_dem_tif_files)
    ds_next_year_dem_full_burst = load_xr_dataset(next_year_dem_tif_files)
    ds_same_year_dem_full_burst = preprocess_dem_xr_dataset(ds_same_year_dem_full_burst, burst_shape)
    ds_next_year_dem_full_burst = preprocess_dem_xr_dataset(ds_next_year_dem_full_burst, burst_shape)

    dem_diff_full_burst_ts = ds_next_year_dem_full_burst.elevation - ds_same_year_dem_full_burst.elevation
    dem_diff_full_burst_da = dem_diff_full_burst_da = dem_diff_full_burst_ts.isel(time=0)  # xarray DataArray

    # ---------------------------------------------------------------------
    # Create TWO subplots side by side
    # ---------------------------------------------------------------------

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

    # === Subplot 1: NRB ===
    im1 = da_same_full_burst.plot.imshow(
        ax=ax1, cmap='bone', vmin=-20, vmax=0, add_colorbar=True
    )

    half = side_length / 2.0

    ax1.set_title(f"ROI over NRB")

    p2  = np.nanpercentile(dem_diff_full_burst_da.values, 2)
    p98 = np.nanpercentile(dem_diff_full_burst_da.values, 98)

    # === Subplot 2: DEM Difference ===
    im2 = dem_diff_full_burst_da.plot.imshow(
        ax=ax2, cmap='RdBu', add_colorbar=True, vmin=p2, vmax=p98
    )
    ax2.set_title(f"ROI over DEM Elevation Difference [2022 - 2021] (m)")

    # add rectangles
    for j,roi in enumerate(burst_roi_aoi_centres[burst]):
        lower_left = (roi[0] - half, roi[1] - half)

        # make the first ROI twice, 1 for each plot
        for i in range(0,2):
            rect = Rectangle(
                lower_left,
                side_length,
                side_length,
                fill=False,
                # edgecolor='red' if j==0 else 'blue',
                edgecolor='green' if j==0 else 'orange',
                linewidth=2,
                # label=f'ROI {j+1}'
                label=f'ROI {j+3}'
            )
            if i == 0:
                ax1.add_patch(rect)
            else:
                ax2.add_patch(rect)

    ax1.legend()
    ax2.legend()

    ax1.set_xticks([])
    ax1.set_yticks([])
    ax2.set_xticks([])
    ax2.set_yticks([])



    plt.suptitle(burst)
    plt.tight_layout()
    plt.show()

## Plot NRB Comparison

In [ ]:
if burst_roi:
    figsize = (6,14)
else:
    figsize = (12, 14)

# Short-hands for the three DataArrays to plot
da_same_nrb = ds_same_year_nrb.isel(time=0)[f'HH_{cal}_filtered_db']
da_next_nrb = ds_next_year_nrb.isel(time=0)[f'HH_{cal}_filtered_db']
da_diff_nrb = nrb_diff_ds.isel(time=0)[f'HH_{cal}_filtered_db_diff']

# Create vertically-stacked subplots
fig, axes = plt.subplots(nrows=3, ncols=1, figsize=figsize, constrained_layout=True)

# 1) Same year
im1 = da_same_nrb.plot.imshow(
    ax=axes[0],
    cmap="bone",
    vmin=-20, vmax=0,
    add_colorbar=False
)
axes[0].set_title("Gamma0 NRB - Same Year DEM (2021)")
axes[0].set_xlabel("")  # optional: declutter
axes[0].set_ylabel("")

cbar1 = fig.colorbar(im1, ax=axes[0], orientation="vertical", fraction=0.046, pad=0.04)
cbar1.set_label("dB")

# 2) Next year
im2 = da_next_nrb.plot.imshow(
    ax=axes[1],
    cmap="bone",
    vmin=-20, vmax=0,
    add_colorbar=False
)
axes[1].set_title("Gamma0 NRB - Next Year DEM (2022)")
axes[1].set_xlabel("")
axes[1].set_ylabel("")

cbar2 = fig.colorbar(im2, ax=axes[1], orientation="vertical", fraction=0.046, pad=0.04)
cbar2.set_label("dB")

# 3) Difference
im3 = da_diff_nrb.plot.imshow(
    ax=axes[2],
    cmap="RdBu",
    vmin=-1, vmax=1,
    add_colorbar=False
)
axes[2].set_title("NRB Difference")
axes[2].set_xlabel("")  # add labels if you want them
axes[2].set_ylabel("")

cbar3 = fig.colorbar(im3, ax=axes[2], orientation="vertical", fraction=0.046, pad=0.04)
cbar3.set_label("Δ dB")


plt.show()

## Plot DEM Comparison

In [ ]:
# Short-hands for the three DataArrays you want to plot
da_same_dem = ds_same_year_dem.isel(time=0)[f'elevation']
da_next_dem = ds_next_year_dem.isel(time=0)[f'elevation']
da_diff_dem = dem_diff_ds.isel(time=0)[f'elevation_m_diff']

dem_p2  = np.nanpercentile(da_same_dem.values, 2)
dem_p98 = np.nanpercentile(da_same_dem.values, 98)

dem_diff_p2  = np.nanpercentile(da_diff_dem.values, 2)
dem_diff_p98 = np.nanpercentile(da_diff_dem.values, 98)

# Create vertically-stacked subplots
fig, axes = plt.subplots(nrows=3, ncols=1, figsize=figsize, constrained_layout=True)

# 1) Same year
im1 = da_same_dem.plot.imshow(
    ax=axes[0],
    #cmap="bone",
    vmin=dem_p2, vmax=dem_p98,
    add_colorbar=False
)
axes[0].set_title("REMA DEM 2021")
axes[0].set_xlabel("")  # optional: declutter
axes[0].set_ylabel("")

cbar1 = fig.colorbar(im1, ax=axes[0], orientation="vertical", fraction=0.046, pad=0.04)
cbar1.set_label("elevation (m)")

# 2) Next year
im2 = da_next_dem.plot.imshow(
    ax=axes[1],
    #cmap="bone",
    vmin=dem_p2, vmax=dem_p98,
    add_colorbar=False
)
axes[1].set_title("REMA DEM 2022")
axes[1].set_xlabel("")
axes[1].set_ylabel("")

cbar2 = fig.colorbar(im2, ax=axes[1], orientation="vertical", fraction=0.046, pad=0.04)
cbar2.set_label("elevation (m)")

# 3) Difference
im3 = da_diff_dem.plot.imshow(
    ax=axes[2],
    cmap="RdBu",
    #vmin=dem_diff_p2, vmax=dem_diff_p98,
    add_colorbar=False
)
axes[2].set_title("Elevation Difference (m)")
axes[2].set_xlabel("")  # add labels if you want them
axes[2].set_ylabel("")

cbar3 = fig.colorbar(im3, ax=axes[2], orientation="vertical", fraction=0.046, pad=0.04)
cbar3.set_label("Δ metres")


plt.show()

## Timeseries of radiometry and elevation

In [ ]:
same_year_nrb_mean_db = ds_same_year_nrb[f'HH_{cal}_db'].mean(dim=["y", "x"], skipna=True)
timseries_nrb_mean_db = ds_next_year_nrb[f'HH_{cal}_db'].mean(dim=["y", "x"], skipna=True)
same_year_elevation_mean = ds_same_year_dem.elevation.mean(dim=["y", "x"])
next_year_elevation_mean = ds_next_year_dem.elevation.mean(dim=["y", "x"])
same_year_slope_mean = ds_same_year_dem.slope.mean(dim=["y", "x"])
next_year_slope_mean = ds_next_year_dem.slope.mean(dim=["y", "x"])
same_year_aspect_mean = ds_same_year_dem.aspect.mean(dim=["y", "x"])
next_year_aspect_mean = ds_next_year_dem.aspect.mean(dim=["y", "x"])

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# --- Left: NRB mean comparison ---
print(f'Plotting NRB')
same_year_nrb_mean_val = np.mean(same_year_nrb_mean_db.values)
ts_nrb_mean_val = np.mean(timseries_nrb_mean_db.values)
print(f'same_year Mean (dB): {same_year_nrb_mean_val}')
print(f'next_year Mean (dB): {ts_nrb_mean_val}')
axes[0][0].plot(same_year_nrb_mean_db.time, same_year_nrb_mean_db, marker='o', label=f'same_year NRB (u={same_year_nrb_mean_val:.4g})')
axes[0][0].plot(timseries_nrb_mean_db.time, timseries_nrb_mean_db, marker='x', label=f'next_year NRB (u={ts_nrb_mean_val:.4g})')
axes[0][0].set_title("NRB Mean Backscatter Over Time", color='red')
axes[0][0].set_xlabel("Time")
axes[0][0].set_ylabel(f"HH_{cal} (dB)")
axes[0][0].legend()
axes[0][0].grid(True)

# --- Right: elevation mean comparison ---
print(f'PLotting elevation')
axes[0][1].plot(same_year_elevation_mean.time, same_year_elevation_mean, marker='o', label='same_year DEM')
axes[0][1].plot(next_year_elevation_mean.time, next_year_elevation_mean, marker='x', label='next_year DEM')
axes[0][1].set_title("DEM Mean Elevation Over Time")
axes[0][1].set_xlabel("Time")
axes[0][1].set_ylabel("Elevation (m)")
axes[0][1].legend()
axes[0][1].grid(True)

# --- Right: slope mean comparison ---
print(f'Plotting slope')
axes[1][0].plot(same_year_slope_mean.time, same_year_slope_mean, marker='o', label='same_year DEM')
axes[1][0].plot(next_year_slope_mean.time, next_year_slope_mean, marker='x', label='next_year DEM')
axes[1][0].set_title("DEM Mean Slope Over Time")
axes[1][0].set_xlabel("Time")
axes[1][0].set_ylabel("Slope (degrees)")
axes[1][0].legend()
axes[1][0].grid(True)

# --- Right: aspect mean comparison ---
print(f'Plotting aspect')
axes[1][1].plot(same_year_aspect_mean.time, same_year_aspect_mean, marker='o', label='same_year DEM')
axes[1][1].plot(next_year_aspect_mean.time, next_year_aspect_mean, marker='x', label='next_year DEM')
axes[1][1].set_title("DEM Mean Aspect Over Time")
axes[1][1].set_xlabel("Time")
axes[1][1].set_ylabel("Aspect (degrees)")
axes[1][1].legend()
axes[1][1].grid(True)

plt.suptitle(f'Burst ID : {burst}')
plt.tight_layout()
plt.savefig(f'{results_folder}/{burst}_linear_plots.png')
plt.show()

## Offset NRB Gif

In [ ]:
band = f'HH_{cal}_filtered_db'

da_2021 = da_same_nrb.expand_dims(time=[pd.Timestamp('2021-01-01')])
da_2022 = da_next_nrb.expand_dims(time=[pd.Timestamp('2022-01-01')])
da_both = xr.concat([da_2021, da_2022], dim='time')
da_both = da_both.odc.assign_crs('EPSG:3031')

#set the DataArray's name (this will become the variable name in the Dataset)
da_both.name = band

# ensure there is no coordinate with the SAME name as the variable
if band in da_both.coords:
    # Drop the clashing coordinate if it’s not needed as a coord
    da_both = da_both.reset_coords(names=band, drop=True)

da_both = da_both.sortby('time')
# Convert to Dataset (xr_animation expects a Dataset)
ds_both = da_both.to_dataset(name=band)

xr_animation(
            ds_both,
            bands=band,
            output_path=f'{results_folder}/{burst}_nrb_geolocation_offset_from_same_vs_next_year_dem_{band}.gif',
            width_pixels=700,
            interval=500,
            show_date='%Y',
            show_text=f'NRB -  Geolocation offset from same vs next year DEM',
            show_colorbar=True,
            imshow_kwargs={"cmap":"bone", "vmin":-20, "vmax":0},
            colorbar_kwargs={'colors': 'black'},
        )

## Offset DEM GIF

In [ ]:
band = f'elevation'

da_2021_dem = da_same_dem.expand_dims(time=[pd.Timestamp('2021-01-01')])
da_2022_dem = da_next_dem.expand_dims(time=[pd.Timestamp('2022-01-01')])
da_both_dem = xr.concat([da_2021_dem, da_2022_dem], dim='time')
da_both_dem = da_both_dem.odc.assign_crs('EPSG:3031')

#set the DataArray's name (this will become the variable name in the Dataset)
da_both_dem.name = band

# ensure there is no coordinate with the SAME name as the variable
if band in da_both_dem.coords:
    # Drop the clashing coordinate if it’s not needed as a coord
    da_both_dem = da_both_dem.reset_coords(names=band, drop=True)

da_both_dem = da_both_dem.sortby('time')
# Convert to Dataset (xr_animation expects a Dataset)
da_both_dem = da_both_dem.to_dataset(name=band)

xr_animation(
            da_both_dem,
            bands=band,
            output_path=f'{results_folder}/{burst}_DEM_geolocation_offset_same_vs_next_year_{band}.gif',
            width_pixels=700,
            interval=500,
            show_date='%Y',
            show_text=f'DEM -  Geolocation Offset 2021 vs 2022 REMA DEM',
            show_colorbar=True,
            imshow_kwargs={"vmin":dem_p2, "vmax":dem_p98},
            colorbar_kwargs={'colors': 'black'},
        )

## Optical Flow

In [ ]:
def run_optical_flow(ds, downscale_factor, baseline='dynamic'):
    ds_course = ds.coarsen(x=downscale_factor, y=downscale_factor, boundary="trim").mean()
    ds_flow = xr_optical_flow(ds_course[f'HH_{cal}_filtered_db'].fillna(0), method="tvl1", baseline=baseline, rescale_units=True)
    print(ds_flow)
    valid_mask = ds.notnull().all(dim="time")
    # ds_flow = ds_flow.where(valid_mask)
    # print(ds_flow)
    # Compute timestep means
    mean_mag = ds_flow.magnitude.mean(dim=("x", "y")) #*20*downscale_factor
    mean_u   = ds_flow.u.mean(dim=("x", "y")) #*20*downscale_factor
    mean_v   = ds_flow.v.mean(dim=("x", "y")) #*20*downscale_factor
    # Compute timestep means
    med_mag = ds_flow.magnitude.median(dim=("x", "y")) #*20*downscale_factor
    med_u   = ds_flow.u.median(dim=("x", "y")) #*20*downscale_factor
    med_v   = ds_flow.v.median(dim=("x", "y")) #*20*downscale_factor
    # Build a clean table
    df = xr.Dataset({
        "time": ds_flow.time,
        "mean magnitude": mean_mag,
        "mean u": mean_u,
        "mean v": mean_v,
        "median magnitude": med_mag,
        "median u": med_u,
        "median v": med_v
    }).to_dataframe().reset_index().drop(columns=["band", "spatial_ref"])
    return df, ds_flow

def run_optical_flow_between_timeseries(ds1, ds2, var=f"HH_{cal}_filtered_db", downscale_factor=3):
    """
    Run optical flow between matching timesteps of ds1 and ds2.
    
    Parameters
    ----------
    ds1, ds2 : xarray.Dataset
        Input datasets with the same spatial dimensions and times.
    var : str
        Variable name to use for optical flow.
    downscale_factor : int
        Factor for coarsening the data.
    
    Returns
    -------
    ds_flow_all : xarray.Dataset
        Optical flow results concatenated along 'time'.
    """

    times = ds1.time.values
    flow_list = []

    for i,t in enumerate(times):
        # Extract single timestep from each dataset
        print(f"time {i+1} of {len(times)} : {t}")
        da1 = ds1[var].sel(time=t)
        da2 = ds2[var].sel(time=t)
        # Coarsen both arrays
        da1_c = da1.coarsen(x=downscale_factor, y=downscale_factor, boundary="trim").mean()
        da2_c = da2.coarsen(x=downscale_factor, y=downscale_factor, boundary="trim").mean()
        # Combine into a 2-step DataArray along a new 'time' dimension
        da_pair = xr.concat([da1_c, da2_c], dim="time")
        # Compute optical flow between first and second step
        ds_flow = xr_optical_flow(da_pair, baseline="first", method="tvl1", rescale_units=True)
        # Assign original timestep as coordinate
        ds_flow = ds_flow.assign_coords(time=[t])
        valid_mask = da_pair.notnull().all(dim="time")
        ds_flow = ds_flow.where(valid_mask)
        flow_list.append(ds_flow)

    # Combine all timesteps into one Dataset
    ds_flow_all = xr.concat(flow_list, dim="time")
     # Compute timestep means
    mean_mag = ds_flow_all.magnitude.mean(dim=("x", "y")) #*20*downscale_factor
    mean_u   = ds_flow_all.u.mean(dim=("x", "y")) #*20*downscale_factor
    mean_v   = ds_flow_all.v.mean(dim=("x", "y")) #*20*downscale_factor
    # Compute timestep means
    med_mag = ds_flow_all.magnitude.median(dim=("x", "y")) #*20*downscale_factor
    med_u   = ds_flow_all.u.median(dim=("x", "y")) #*20*downscale_factor
    med_v   = ds_flow_all.v.median(dim=("x", "y")) #*20*downscale_factor
    # Build a clean table
    df = xr.Dataset({
        "time": ds_flow_all.time,
        "mean magnitude": mean_mag,
        "mean u": mean_u,
        "mean v": mean_v,
        "median magnitude": med_mag,
        "median u": med_u,
        "median v": med_v
    }).to_dataframe().reset_index().drop(columns=["band", "spatial_ref"])
    return df, ds_flow_all

def plot_optical_flow(
        ds_flow_to_plot, 
        base_plot_arr, 
        savepath, 
        year_label="",
        base_plot_label="",
        base_plot_cmap="RdBu",
        vrange=[],
        keysize_m = 100,
        VECTOR_EXAGGERATION = 20,  # map metres per (m/year)
        arrow_density_coarsen = {"x": 35, "y": 35},
        figsize=(10, 5)
        ):
    fig, ax = plt.subplots(1, 1, figsize=figsize)
    # Sub-sample array for clearer quiver plot
    ds_flow_coarse = ds_flow_to_plot.coarsen(arrow_density_coarsen, boundary="trim").median()
    # Flip vertical axis to avoid xarray issue where negative Y
    # coordinates cause vectors to be inverted
    ds_flow_coarse["v"] = -ds_flow_coarse.v
    # Plot background rate of vertical change image
    base_plot_arr.plot(
        ax=ax,
        cmap=base_plot_cmap,
        cbar_kwargs={"label": base_plot_label},
        vmin=None if not vrange else vrange[0],
        vmax=None if not vrange else vrange[1],
    )
    # Add a basemap for context
    ctx.add_basemap(
        ax,
        source=ctx.providers.Esri.WorldImagery,
        crs="EPSG:3031",
        attribution="Esri WorldImagery",
        attribution_size=1,
        alpha=0.7,
    )

    # Add quiver plot directly from xarray
    quiver = ds_flow_coarse.plot.quiver(
        x="x",
        y="y",
        u="u",
        v="v",
        ax=ax,
        color="black",
        pivot="mid",
        add_guide=False,
        width=0.0015,
        scale_units="xy",
        angles="xy",
        scale=1/VECTOR_EXAGGERATION,
    )
    # Add key and plot title
    ax.quiverkey(quiver, 0.85, 0.85, keysize_m, f"{keysize_m}m / year")
    # add test to plot for year
    ax.text(
        0.85, 0.95,                 # just below the quiver key
        year_label,
        transform=ax.transAxes,
        ha="center",
        va="top",
        fontsize=14
    )

    ax.set_title(f"Burst ID : {burst}, Surface flow (optical flow vectors)")
    ax.set_axis_off()
    fig.tight_layout();
    plt.savefig(savepath);

def plot_optical_flow_for_year(ds, base_plot_arr, year, savepath, base_plot_label, base_plot_cmap="RdBu",vrange=[]):
    ds_year = ds.sel(time=str(year))
    ds_year_median = ds_year.median(dim="time")
    plot_optical_flow(ds_year_median, base_plot_arr, savepath, year_label=f"{year}",base_plot_label=base_plot_label,base_plot_cmap=base_plot_cmap, vrange=vrange)


In [ ]:
downscale_factor = 5 # factor to downsample the image by i.e. 20m -> 60m (x3)
print(f'Calculating annual velocity with optical flow for static REMA')
df_nrb_flow_summary, ds_nrb_flow = run_optical_flow(ds_both, downscale_factor)

In [ ]:
df_nrb_flow_summary.round(1).drop(columns=['time'])

## Median Plots

In [ ]:
print(f'making plots')
savepath = f'{results_folder}/{burst}_REMA_10_median_optical_flow.png'
base_label = f"Elevation Difference (m)"
dem_type = 'REMA_10'
plot_optical_flow(
    ds_nrb_flow.median(dim="time"), 
    da_diff_dem,
    #da_diff_nrb,
    savepath, 
    year_label="",
    base_plot_label=base_label,
    vrange=[],
    keysize_m = 10,
    VECTOR_EXAGGERATION = 10,  # map metres per (m/year)
    arrow_density_coarsen = {"x": 20, "y": 20},
    figsize=(6, 5)
    )
df_nrb_flow_summary.to_csv(savepath.replace('png','csv'))